In [1]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import time
import warnings
from pathlib import Path
from tqdm import tqdm
from copy import deepcopy
import random
import sys
from scipy import stats
import psutil
import tracemalloc
import gc
import json as json_lib

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset

from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, confusion_matrix, f1_score,
                             precision_score, recall_score, classification_report)

# Standard feature selection imports
from sklearn.feature_selection import mutual_info_classif, RFE, SelectKBest
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression

# Import ALL EvoloPy optimizers
sys.path.append('../EvoloPy-master')
from EvoloPy.optimizers import BAT, CS, DE, FFA, GA, GWO, HHO, JAYA, MFO, MVO, PSO, SCA, SSA, WOA

warnings.filterwarnings('ignore')
np.random.seed(42)
torch.manual_seed(42)
random.seed(42)

# Global device
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
if torch.cuda.is_available():
    print(f"PyTorch version: {torch.__version__}")
    print(f"GPU: {torch.cuda.get_device_name(0)}")


Device: cuda
PyTorch version: 2.6.0+cu124
GPU: NVIDIA GeForce GTX 1080 Ti


## Configuration

In [2]:
# Paths
FEATURE_DIR = Path("features_noise_1.0_50")

# Master output directory
RESULTS_ROOT = Path("results_noise_1.0_50")
RESULTS_ROOT.mkdir(exist_ok=True)
PLOTS_DIR = RESULTS_ROOT / "plots"
PLOTS_DIR.mkdir(exist_ok=True)

# Hyperparameters
N_FOLDS = 10
N_EPOCHS = 300
BATCH_SIZE = 32
LEARNING_RATE = 0.001
HIDDEN_DIM = 128

# Feature selection hyperparameters - metaheuristic
N_POPULATION = 20
MAX_ITERATIONS = 30

# For standard methods (target: select ~50% of features)
TARGET_FEATURE_PERCENTAGE = 0.5

# All EvoloPy optimizer names and their callable entry points
EVOLOPY_OPTIMIZERS = {
    'BAT': BAT.BAT,
    'CS':  CS.CS,
    'DE':  DE.DE,
    'FFA': FFA.FFA,
    'GA':  GA.GA,
    'GWO': GWO.GWO,
    'HHO': HHO.HHO,
    'JAYA': JAYA.JAYA,
    'MFO': MFO.MFO,
    'MVO': MVO.MVO,
    'PSO': PSO.PSO,
    'SCA': SCA.SCA,
    'SSA': SSA.SSA,
    'WOA': WOA.WOA,
}

ALL_METHODS = ['baseline', 'mutual_info', 'rfe', 'lasso'] + [f'meta_{k}' for k in EVOLOPY_OPTIMIZERS.keys()]

# Create per-method subdirectories
for method in ALL_METHODS:
    (RESULTS_ROOT / method).mkdir(exist_ok=True)

print(f"Configuration:")
print(f"  N_FOLDS: {N_FOLDS}")
print(f"  N_EPOCHS: {N_EPOCHS}")
print(f"  Target feature selection: {TARGET_FEATURE_PERCENTAGE*100:.0f}%")
print(f"  Metaheuristic pop: {N_POPULATION}, iter: {MAX_ITERATIONS}")
print(f"  EvoloPy optimizers: {list(EVOLOPY_OPTIMIZERS.keys())}")
print(f"  Total methods (incl. baseline): {len(ALL_METHODS)}")


Configuration:
  N_FOLDS: 10
  N_EPOCHS: 300
  Target feature selection: 50%
  Metaheuristic pop: 20, iter: 30
  EvoloPy optimizers: ['BAT', 'CS', 'DE', 'FFA', 'GA', 'GWO', 'HHO', 'JAYA', 'MFO', 'MVO', 'PSO', 'SCA', 'SSA', 'WOA']
  Total methods (incl. baseline): 18


## Load Data

In [3]:
# Load features
X_feat = joblib.load(FEATURE_DIR / "X_feat.pkl")
y = np.load(FEATURE_DIR / "y.npy")
subjects = np.load(FEATURE_DIR / "participants.npy")
le = joblib.load(FEATURE_DIR / "label_encoder.pkl")
print(f"Loaded {len(X_feat)} samples")
print(f"Number of classes: {len(np.unique(y))}")
print(f"Number of subjects: {len(np.unique(subjects))}")

# Get feature dimensions from first sample
first_sample = X_feat[0]
MODALITY_KEYS = ['sensor_feat', 'skeleton_feat']
MODALITY_NAMES = ['sensor', 'skeleton']

RAW_FEATURE_DIMS = {}
for key, name in zip(MODALITY_KEYS, MODALITY_NAMES):
    RAW_FEATURE_DIMS[name] = first_sample[key].shape[0]

# Store per-modality arrays (N_samples x D_modality)
X_per_modality = {}

for key, name in zip(MODALITY_KEYS, MODALITY_NAMES):
    lengths = [len(s[key]) for s in X_feat if s[key] is not None]
    print(name, "unique lengths:", set(lengths))

for key, name in zip(MODALITY_KEYS, MODALITY_NAMES):
    X_per_modality[name] = np.array([s[key] for s in X_feat])

print(f"\nRaw feature dimensions:")
for name, dim in RAW_FEATURE_DIMS.items():
    print(f"  {name}: {dim}")
print(f"  TOTAL: {sum(RAW_FEATURE_DIMS.values())}")


Loaded 616 samples
Number of classes: 10
Number of subjects: 10
sensor unique lengths: {4480}
skeleton unique lengths: {1879}

Raw feature dimensions:
  sensor: 4480
  skeleton: 1879
  TOTAL: 6359


## Neural Network Model (Same as Original)

In [4]:
class MultiModalDataset(Dataset):
    def __init__(self, features, labels):
        self.features = torch.FloatTensor(features)
        self.labels = torch.LongTensor(labels)
    
    def __len__(self):
        return len(self.labels)
    
    def __getitem__(self, idx):
        return self.features[idx], self.labels[idx]


class SimpleNN(nn.Module):
    """MLP for unified feature vector (adaptive to feature subset size)"""
    def __init__(self, input_dim, num_classes):
        super().__init__()
        
        # Adaptive hidden layer sizing based on input dimension
        hidden1 = max(128, min(512, input_dim * 2))
        hidden2 = max(64, min(256, hidden1 // 2))
        
        self.classifier = nn.Sequential(
            nn.Linear(input_dim, hidden1),
            nn.BatchNorm1d(hidden1),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(hidden1, hidden2),
            nn.BatchNorm1d(hidden2),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(hidden2, num_classes)
        )
    
    def forward(self, x):
        return self.classifier(x)

print("Neural network model defined")


Neural network model defined


## Helper Functions

In [5]:
def prepare_fold_data_per_modality(X_per_modality, train_idx, val_idx, test_idx):
    """
    Returns:
        X_train, X_val, X_test: normalized concatenated arrays
        feature_dims: dict of {modality: dim}
    """
    modality_train = {}
    modality_val = {}
    modality_test = {}
    scalers = {}
    feature_dims = {}
    
    for name in MODALITY_NAMES:
        X_mod = X_per_modality[name]
        
        # Split
        X_tr = X_mod[train_idx]
        X_v  = X_mod[val_idx]
        X_te = X_mod[test_idx]
        
        # Per-modality normalization (fit on train only)
        scaler = StandardScaler()
        X_tr = scaler.fit_transform(X_tr)
        X_v  = scaler.transform(X_v)
        X_te = scaler.transform(X_te)
        scalers[name] = scaler
        
        modality_train[name] = X_tr
        modality_val[name]   = X_v
        modality_test[name]  = X_te
        feature_dims[name]   = X_tr.shape[1]
    
    # Concatenate: sensor | skeleton
    X_train = np.concatenate([modality_train[n] for n in MODALITY_NAMES], axis=1)
    X_val   = np.concatenate([modality_val[n]   for n in MODALITY_NAMES], axis=1)
    X_test  = np.concatenate([modality_test[n]  for n in MODALITY_NAMES], axis=1)
    
    total = sum(feature_dims.values())
    print(f"    Feature dims after processing: " + 
          " | ".join(f"{n}={feature_dims[n]}" for n in MODALITY_NAMES) +
          f" | TOTAL={total}")
    
    return X_train, X_val, X_test, feature_dims


def prepare_unified_features(X_feat_list, feature_mask=None):
    """Concatenate all modality features into unified vector (legacy, used for masks)"""
    unified_features = []
    
    for sample in X_feat_list:
        feat_vector = np.concatenate([
            sample['sensor_feat'],
            sample['skeleton_feat']
        ])
        
        if feature_mask is not None:
            feat_vector = feat_vector[feature_mask]
        
        unified_features.append(feat_vector)
    
    return np.array(unified_features)


def calculate_modality_retention(binary_mask, feature_dims):
    """Calculate how many features retained per modality"""
    start_idx = 0
    retention = {}
    
    for modality in MODALITY_NAMES:
        dim = feature_dims[modality]
        end_idx = start_idx + dim
        modality_mask = binary_mask[start_idx:end_idx]
        num_selected = np.sum(modality_mask)
        percentage = (num_selected / dim) * 100
        
        retention[modality] = {
            'selected': int(num_selected),
            'total': dim,
            'percentage': percentage
        }
        start_idx = end_idx
    
    return retention


def train_and_evaluate(model, train_loader, val_loader, test_loader, num_epochs, lr):
    """Train model and return metrics"""
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    
    model.train()
    for epoch in range(num_epochs):
        for features, labels in train_loader:
            features = features.to(DEVICE)
            labels = labels.to(DEVICE)
            
            optimizer.zero_grad()
            outputs = model(features)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
    

    # Evaluate
    model.eval()
    with torch.no_grad():
        # Validation
        val_preds, val_true = [], []
        for features, labels in val_loader:
            features = features.to(DEVICE)
            outputs = model(features)
            preds = torch.argmax(outputs, dim=1).cpu().numpy()
            val_preds.extend(preds)
            val_true.extend(labels.numpy())
        val_acc = accuracy_score(val_true, val_preds)
        
        # Test
        test_preds, test_true = [], []
        for features, labels in test_loader:
            features = features.to(DEVICE)
            outputs = model(features)
            preds = torch.argmax(outputs, dim=1).cpu().numpy()
            test_preds.extend(preds)
            test_true.extend(labels.numpy())
        test_acc = accuracy_score(test_true, test_preds)
    
    return val_acc, test_acc


def count_model_parameters(model):
    """Count trainable parameters in a model"""
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

def get_model_size_mb(model):
    """Get model size in MB"""
    param_size = sum(p.nelement() * p.element_size() for p in model.parameters())
    buffer_size = sum(b.nelement() * b.element_size() for b in model.buffers())
    return (param_size + buffer_size) / (1024 ** 2)

def get_gpu_memory_mb():
    """Get current GPU memory usage in MB"""
    if torch.cuda.is_available():
        return torch.cuda.memory_allocated() / (1024 ** 2)
    return 0.0

def get_dataset_size_mb(X):
    """Get dataset size in MB"""
    return X.nbytes / (1024 ** 2)

print("Helper functions defined (with per-modality normalization)")


Helper functions defined (with per-modality normalization)


## Enhanced Evaluation (returns full metrics per fold)

In [6]:
def train_and_evaluate_full(model, train_loader, val_loader, test_loader, num_epochs, lr, num_classes):
    """Train model and return comprehensive metrics including predictions for confusion matrix"""
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    
    # Track GPU memory before training
    gpu_mem_before = get_gpu_memory_mb()
    train_start = time.time()

    train_losses = []
    best_val_acc = -1.0
    best_model_state = None
    
    for epoch in range(num_epochs):
        model.train()
        epoch_loss = 0.0

        for features, labels in train_loader:
            features = features.to(DEVICE)
            labels = labels.to(DEVICE)
            optimizer.zero_grad()
            outputs = model(features)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()

        train_losses.append(epoch_loss / len(train_loader))

        # Check validation accuracy after each epoch
        model.eval()
        val_correct, val_total = 0, 0
        with torch.no_grad():
            for features, labels in val_loader:
                features = features.to(DEVICE)
                outputs = model(features)
                preds = torch.argmax(outputs, dim=1)
                val_correct += (preds.cpu() == labels).sum().item()
                val_total += labels.size(0)

        epoch_val_acc = val_correct / val_total

        if epoch_val_acc > best_val_acc:
            best_val_acc = epoch_val_acc
            best_model_state = deepcopy(model.state_dict())
    
    train_time = time.time() - train_start
    gpu_mem_after = get_gpu_memory_mb()

    # Restore best model
    model.load_state_dict(best_model_state)
    
    # Final evaluation
    model.eval()
    with torch.no_grad():
        val_preds, val_true = [], []
        for features, labels in val_loader:
            features = features.to(DEVICE)
            outputs = model(features)
            preds = torch.argmax(outputs, dim=1).cpu().numpy()
            val_preds.extend(preds)
            val_true.extend(labels.numpy())
        
        # Test
        test_preds, test_true = [], []
        for features, labels in test_loader:
            features = features.to(DEVICE)
            outputs = model(features)
            preds = torch.argmax(outputs, dim=1).cpu().numpy()
            test_preds.extend(preds)
            test_true.extend(labels.numpy())
    
    val_acc = accuracy_score(val_true, val_preds)
    test_acc = accuracy_score(test_true, test_preds)
    
    metrics = {
        'val_acc': val_acc,
        'test_acc': test_acc,
        'test_f1_macro': f1_score(test_true, test_preds, average='macro', zero_division=0),
        'test_f1_weighted': f1_score(test_true, test_preds, average='weighted', zero_division=0),
        'test_precision_macro': precision_score(test_true, test_preds, average='macro', zero_division=0),
        'test_recall_macro': recall_score(test_true, test_preds, average='macro', zero_division=0),
        'test_preds': np.array(test_preds),
        'test_true': np.array(test_true),
        'val_preds': np.array(val_preds),
        'val_true': np.array(val_true),
        'train_time_sec': train_time,
        'gpu_mem_before_mb': gpu_mem_before,
        'gpu_mem_after_mb': gpu_mem_after,
        'gpu_mem_peak_mb': torch.cuda.max_memory_allocated() / (1024**2) if torch.cuda.is_available() else 0,
        'model_params': count_model_parameters(model),
        'model_size_mb': get_model_size_mb(model),
        'train_losses': train_losses,
    }
    return metrics

print("Enhanced evaluation function defined")


Enhanced evaluation function defined


## Feature Selection Methods
### 0. EvoloPy Metaheuristics

In [7]:
# ============================================================================
# EXACT EVOLOPY BAT IMPLEMENTATION FROM ORIGINAL CODE
# ============================================================================

def train_model_quick(model, train_loader, val_loader, epochs, lr, device):
    """Quick training for fitness evaluation"""
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    
    best_val_loss = float('inf')
    best_val_acc = 0.0
    
    for epoch in range(epochs):
        model.train()
        for features, labels in train_loader:
            features, labels = features.to(device), labels.to(device)
            optimizer.zero_grad()
            loss = criterion(model(features), labels)
            loss.backward()
            optimizer.step()
        
        model.eval()
        val_loss, val_correct, val_total = 0.0, 0, 0
        with torch.no_grad():
            for features, labels in val_loader:
                features, labels = features.to(device), labels.to(device)
                outputs = model(features)
                val_loss += criterion(outputs, labels).item()
                val_correct += (torch.argmax(outputs, 1) == labels).sum().item()
                val_total += labels.size(0)
        
        val_loss /= len(val_loader)
        val_acc = val_correct / val_total
        if val_loss < best_val_loss:
            best_val_loss, best_val_acc = val_loss, val_acc
    
    return best_val_loss, best_val_acc

# ============================================================================
# FITNESS FUNCTION: Using (1 - accuracy)
# ============================================================================
# Rationale: (1 - accuracy) is preferred over loss because:
#   - It directly optimizes the metric we care about (accuracy)
#   - It is bounded in [0, 1], making it cleaner for metaheuristic optimization
#   - Loss can vary in scale across different feature subsets
#   - Accuracy-based fitness is more interpretable and comparable across methods
#   - Less sensitive to calibration issues of the neural network

def create_fitness_function_evolopy(X_train, y_train, X_val, y_val, num_classes, total_features=None):
    """Create fitness function for EvoloPy using (1 - accuracy) + feature penalty"""
    eval_count = [0]  # track evaluations
    
    def fitness_function(binary_mask):
        try:
            if binary_mask.dtype != bool:
                binary_mask = binary_mask > 0.5

            num_selected = np.sum(binary_mask)
            if num_selected == 0:
                return 1.0
            
            X_tr_sel = X_train[:, binary_mask]
            X_val_sel = X_val[:, binary_mask]
            
            train_dataset = MultiModalDataset(X_tr_sel, y_train)
            val_dataset = MultiModalDataset(X_val_sel, y_val)
            train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
            val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE)
            
            model = SimpleNN(X_tr_sel.shape[1], num_classes).to(DEVICE)
            
            val_loss, val_acc = train_model_quick(
                model, train_loader, val_loader,
                epochs=30, lr=1e-3, device=DEVICE
            )
            
            del model, train_dataset, val_dataset, train_loader, val_loader
            torch.cuda.empty_cache()
            
            eval_count[0] += 1
            
            # Weighted fitness: accuracy + feature reduction pressure
            alpha = 0.95  # weight for accuracy
            beta = 0.05   # weight for feature reduction
            feature_ratio = num_selected / len(binary_mask)

            fitness = alpha * (1.0 - val_acc) + beta * feature_ratio
            return fitness
            
        except Exception as e:
            print(f"Error in fitness: {e}")
            return 1.0
        
    return fitness_function


def s_transfer(x):
    return 1 / (1 + np.exp(-10 * (x - 0.5)))


def run_evolopy_optimizer(optimizer_name, optimizer_func, X_train, y_train, X_val, y_val, num_classes, total_features=None, feature_dims=None):
    """Run any EvoloPy optimizer generically"""
    print(f"    Running EvoloPy {optimizer_name}...")
    print(f"      Fitness: (1 - accuracy), Pop: {N_POPULATION}, Iter: {MAX_ITERATIONS}")
    
    n_feats = total_features if total_features is not None else X_train.shape[1]
    fitness_func = create_fitness_function_evolopy(X_train, y_train, X_val, y_val, num_classes, n_feats)
    
    start = time.time()
    solution = optimizer_func(fitness_func, 0, 1, n_feats, N_POPULATION, MAX_ITERATIONS)
    exec_time = time.time() - start
    
    binary_mask = solution.bestIndividual > 0.5
    feat_dims = feature_dims if feature_dims is not None else FEATURE_DIMS
    modality_ret = calculate_modality_retention(binary_mask, feat_dims)
    
    results = {
        'mask': binary_mask,
        'convergence': solution.convergence.tolist() if hasattr(solution.convergence, 'tolist') else list(solution.convergence),
        'best_fitness': float(solution.convergence[-1]) if len(solution.convergence) > 0 else float('inf'),
        'execution_time': exec_time,
        'num_selected': int(np.sum(binary_mask)),
        'num_total': len(binary_mask),
        'modality_retention': modality_ret,
        'method': f'meta_{optimizer_name}'
    }
    
    print(f"      Selected: {results['num_selected']}/{results['num_total']} "
          f"({results['num_selected']/results['num_total']*100:.1f}%), "
          f"Fitness: {results['best_fitness']:.4f}, Time: {exec_time:.1f}s")
    print(f"      Modality: "
          f"S={modality_ret['sensor']['percentage']:.0f}%, "
          f"Sk={modality_ret['skeleton']['percentage']:.0f}%, ")
    
    return results

print("All EvoloPy metaheuristic runners defined")
print(f"Available optimizers: {list(EVOLOPY_OPTIMIZERS.keys())}")


All EvoloPy metaheuristic runners defined
Available optimizers: ['BAT', 'CS', 'DE', 'FFA', 'GA', 'GWO', 'HHO', 'JAYA', 'MFO', 'MVO', 'PSO', 'SCA', 'SSA', 'WOA']


### 1-3. Standard Feature Selection Methods (Mutual Info, RFE, LASSO)

In [8]:
def run_mutual_info_fs(X_train, y_train, X_val, y_val, num_classes, total_features=None, feature_dims=None):
    """Run Mutual Information feature selection"""
    print("    Running Mutual Information...")
    
    start_time = time.time()
    
    # Calculate mutual information scores
    mi_scores = mutual_info_classif(X_train, y_train, random_state=42)
    
    # Select top k features (target percentage)
    n_feats = total_features if total_features is not None else TOTAL_FEATURES
    k = int(n_feats * TARGET_FEATURE_PERCENTAGE)
    top_k_indices = np.argsort(mi_scores)[::-1][:k]
    
    binary_mask = np.zeros(n_feats, dtype=bool)
    binary_mask[top_k_indices] = True
    
    execution_time = time.time() - start_time
    
    feat_dims = feature_dims if feature_dims is not None else FEATURE_DIMS
    modality_retention = calculate_modality_retention(binary_mask, feat_dims)
    
    results = {
        'mask': binary_mask,
        'execution_time': execution_time,
        'num_selected': int(np.sum(binary_mask)),
        'num_total': len(binary_mask),
        'modality_retention': modality_retention,
        'mi_scores': mi_scores,
        'method': 'mutual_info'
    }
    
    print(f"      Selected: {results['num_selected']}/{results['num_total']} "
          f"({results['num_selected']/results['num_total']*100:.1f}%), "
          f"Time: {results['execution_time']:.1f}s")
    
    return results


def run_rfe_fs(X_train, y_train, X_val, y_val, num_classes, total_features=None, feature_dims=None):
    """Run RFE feature selection"""
    print("    Running RFE...")
    
    start_time = time.time()
    
    # Use Random Forest as base estimator
    estimator = RandomForestClassifier(n_estimators=50, random_state=42, n_jobs=-1)
    
    # Select top k features
    n_feats = total_features if total_features is not None else TOTAL_FEATURES
    k = int(n_feats * TARGET_FEATURE_PERCENTAGE)
    selector = RFE(estimator, n_features_to_select=k, step=50)  # Remove 50 features at a time
    selector.fit(X_train, y_train)
    
    binary_mask = selector.support_
    
    execution_time = time.time() - start_time
    
    feat_dims = feature_dims if feature_dims is not None else FEATURE_DIMS
    modality_retention = calculate_modality_retention(binary_mask, feat_dims)
    
    results = {
        'mask': binary_mask,
        'execution_time': execution_time,
        'num_selected': int(np.sum(binary_mask)),
        'num_total': len(binary_mask),
        'modality_retention': modality_retention,
        'ranking': selector.ranking_,
        'method': 'rfe'
    }
    
    print(f"      Selected: {results['num_selected']}/{results['num_total']} "
          f"({results['num_selected']/results['num_total']*100:.1f}%), "
          f"Time: {results['execution_time']:.1f}s")
    
    return results


def run_lasso_fs(X_train, y_train, X_val, y_val, num_classes, total_features=None, feature_dims=None):
    """Run LASSO feature selection"""
    print("    Running LASSO...")
    start_time = time.time()

    # Train Lasso to get feature importance scores
    lasso = LogisticRegression(
        penalty='l1',
        C=0.01,
        solver='saga',
        random_state=42,
        max_iter=1000
    )
    lasso.fit(X_train, y_train)

    # Rank features by coefficient magnitude across all classes
    coef_abs = np.abs(lasso.coef_).sum(axis=0)

    # Select top-k features to match target percentage
    n_feats = total_features if total_features is not None else TOTAL_FEATURES
    target_count = int(n_feats * TARGET_FEATURE_PERCENTAGE)
    top_indices = np.argsort(coef_abs)[::-1][:target_count]
    binary_mask = np.zeros(n_feats, dtype=bool)
    binary_mask[top_indices] = True

    execution_time = time.time() - start_time
    feat_dims = feature_dims if feature_dims is not None else FEATURE_DIMS
    modality_retention = calculate_modality_retention(binary_mask, feat_dims)

    results = {
        'mask': binary_mask,
        'execution_time': execution_time,
        'num_selected': int(np.sum(binary_mask)),
        'num_total': len(binary_mask),
        'modality_retention': modality_retention,
        'coefficients': coef_abs,
        'method': 'lasso'
    }

    print(f"      Selected: {results['num_selected']}/{results['num_total']} "
          f"({results['num_selected']/results['num_total']*100:.1f}%), "
          f"Time: {results['execution_time']:.1f}s")

    return results

print("Standard FS methods defined (Mutual Info, RFE, LASSO)")


Standard FS methods defined (Mutual Info, RFE, LASSO)


## Main Per-Fold Experiment Runner

In [9]:
def run_all_methods_per_fold():
    """
    Run ALL methods across ALL folds with:
    1. Per-modality normalization (fit on train only)
    2. Concatenation AFTER normalization
    """
    print("="*80)
    print("STARTING COMPREHENSIVE PER-FOLD EXPERIMENTS")
    print(f"Methods: baseline + 3 standard + {len(EVOLOPY_OPTIMIZERS)} metaheuristics = {len(ALL_METHODS)} total")
    print(f"Folds: {N_FOLDS}")
    print(f"Per-modality normalization: ENABLED")
    print("="*80)
    
    num_classes = len(np.unique(y))
    
    # Master results dict: {method_name: {fold_idx: {...}}}
    master_results = {m: {} for m in ALL_METHODS}
    
    # Build sample indices for GroupKFold
    # We use a dummy X just for splitting indices
    n_samples = len(y)
    dummy_X = np.zeros((n_samples, 1))
    
    gkf = GroupKFold(n_splits=N_FOLDS)
    
    for fold_idx, (train_val_idx, test_idx) in enumerate(gkf.split(dummy_X, y, groups=subjects)):
        print(f"\n{'#'*80}")
        print(f"# FOLD {fold_idx + 1}/{N_FOLDS}")
        print(f"{'#'*80}")
        
        # --- Subject-based val split within train_val ---
        train_val_subjects = np.unique(subjects[train_val_idx])
        val_subject = train_val_subjects[-1]
        
        val_mask_in_tv   = subjects[train_val_idx] == val_subject
        train_mask_in_tv = ~val_mask_in_tv
        
        train_idx = train_val_idx[train_mask_in_tv]
        val_idx   = train_val_idx[val_mask_in_tv]
        
        print(f"  Train: {len(train_idx)}, Val: {len(val_idx)}, Test: {len(test_idx)}")
        
        # =================================================================
        # Per-modality normalization (fit on train only)
        # =================================================================
        X_train, X_val, X_test, fold_feature_dims = prepare_fold_data_per_modality(
            X_per_modality, train_idx, val_idx, test_idx
        )
        
        fold_total_features = sum(fold_feature_dims.values())
        
        # Original dataset size
        orig_dataset_size_mb = get_dataset_size_mb(X_train) + get_dataset_size_mb(X_val) + get_dataset_size_mb(X_test)
        
        # =====================================================================
        # Run each method
        # =====================================================================
        for method_name in ALL_METHODS:
            print(f"\n  --- {method_name.upper()} ---")
            
            fs_start_time = time.time()
            
            # Feature selection
            if method_name == 'baseline':
                feature_mask = np.ones(fold_total_features, dtype=bool)
                fs_results = {
                    'mask': feature_mask,
                    'execution_time': 0,
                    'num_selected': fold_total_features,
                    'num_total': fold_total_features,
                    'method': 'baseline'
                }
            elif method_name == 'mutual_info':
                fs_results = run_mutual_info_fs(X_train, y[train_idx], X_val, y[val_idx], num_classes,
                                                total_features=fold_total_features, feature_dims=fold_feature_dims)
                feature_mask = fs_results['mask']
            elif method_name == 'rfe':
                fs_results = run_rfe_fs(X_train, y[train_idx], X_val, y[val_idx], num_classes,
                                        total_features=fold_total_features, feature_dims=fold_feature_dims)
                feature_mask = fs_results['mask']
            elif method_name == 'lasso':
                fs_results = run_lasso_fs(X_train, y[train_idx], X_val, y[val_idx], num_classes,
                                          total_features=fold_total_features, feature_dims=fold_feature_dims)
                feature_mask = fs_results['mask']
            elif method_name.startswith('meta_'):
                opt_name = method_name.replace('meta_', '')
                opt_func = EVOLOPY_OPTIMIZERS[opt_name]
                fs_results = run_evolopy_optimizer(opt_name, opt_func,
                                                   X_train, y[train_idx], X_val, y[val_idx], num_classes,
                                                   total_features=fold_total_features, feature_dims=fold_feature_dims)
                feature_mask = fs_results['mask']
            else:
                raise ValueError(f"Unknown method: {method_name}")
            
            fs_time = time.time() - fs_start_time
            
            # Apply feature mask
            X_train_sel = X_train[:, feature_mask]
            X_val_sel = X_val[:, feature_mask]
            X_test_sel = X_test[:, feature_mask]
            
            # Dataset size after selection
            opt_dataset_size_mb = get_dataset_size_mb(X_train_sel) + get_dataset_size_mb(X_val_sel) + get_dataset_size_mb(X_test_sel)
            
            # Build and train final model
            model = SimpleNN(X_train_sel.shape[1], num_classes).to(DEVICE)
            
            train_dataset = MultiModalDataset(X_train_sel, y[train_idx])
            val_dataset = MultiModalDataset(X_val_sel, y[val_idx])
            test_dataset = MultiModalDataset(X_test_sel, y[test_idx])
            
            train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
            val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE)
            test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE)
            
            if torch.cuda.is_available():
                torch.cuda.reset_peak_memory_stats()
            
            eval_metrics = train_and_evaluate_full(
                model, train_loader, val_loader, test_loader, N_EPOCHS, LEARNING_RATE, num_classes
            )
            
            # Modality retention
            if method_name != 'baseline':
                modality_ret = fs_results.get('modality_retention', 
                    calculate_modality_retention(feature_mask, fold_feature_dims))
            else:
                modality_ret = {m: {'selected': d, 'total': d, 'percentage': 100.0} 
                               for m, d in fold_feature_dims.items()}
            
            # Compile comprehensive fold result
            fold_result = {
                'fold': fold_idx,
                'method': method_name,
                'num_train': len(train_idx),
                'num_val': len(val_idx),
                'num_test': len(test_idx),
                # Accuracy & classification metrics
                'val_acc': eval_metrics['val_acc'],
                'test_acc': eval_metrics['test_acc'],
                'test_f1_macro': eval_metrics['test_f1_macro'],
                'test_f1_weighted': eval_metrics['test_f1_weighted'],
                'test_precision_macro': eval_metrics['test_precision_macro'],
                'test_recall_macro': eval_metrics['test_recall_macro'],
                # Feature selection info
                'num_features_selected': int(np.sum(feature_mask)),
                'num_features_total': fold_total_features,
                'feature_retention_pct': float(np.sum(feature_mask) / fold_total_features * 100),
                'modality_retention': modality_ret,
                'feature_mask': feature_mask,
                # Feature dims for this fold
                'feature_dims': fold_feature_dims,
                # Timing
                'fs_execution_time': fs_results.get('execution_time', 0),
                'train_time_sec': eval_metrics['train_time_sec'],
                'total_time_sec': fs_time + eval_metrics['train_time_sec'],
                # Model size
                'model_params': eval_metrics['model_params'],
                'model_size_mb': eval_metrics['model_size_mb'],
                # Dataset size
                'original_dataset_size_mb': orig_dataset_size_mb,
                'optimized_dataset_size_mb': opt_dataset_size_mb,
                'dataset_reduction_pct': float((1 - opt_dataset_size_mb / orig_dataset_size_mb) * 100) if orig_dataset_size_mb > 0 else 0,
                # GPU
                'gpu_mem_peak_mb': eval_metrics['gpu_mem_peak_mb'],
                # Convergence (metaheuristics only)
                'convergence': fs_results.get('convergence', None),
                'best_fitness': fs_results.get('best_fitness', None),
            }
            
            master_results[method_name][fold_idx] = fold_result
            
            # Save per-fold result immediately
            fold_dir = RESULTS_ROOT / method_name
            fold_save = {k: v for k, v in fold_result.items() if k not in ['feature_mask']}
            # Convert numpy to python types for JSON
            fold_save_clean = {}
            for k, v in fold_save.items():
                if isinstance(v, np.ndarray):
                    fold_save_clean[k] = v.tolist()
                elif isinstance(v, (np.floating, np.integer)):
                    fold_save_clean[k] = float(v)
                elif isinstance(v, dict):
                    fold_save_clean[k] = {}
                    for kk, vv in v.items():
                        if isinstance(vv, dict):
                            fold_save_clean[k][kk] = {kkk: float(vvv) if isinstance(vvv, (np.floating, np.integer)) else vvv for kkk, vvv in vv.items()}
                        else:
                            fold_save_clean[k][kk] = float(vv) if isinstance(vv, (np.floating, np.integer)) else vv
                else:
                    fold_save_clean[k] = v
            
            with open(fold_dir / f"fold_{fold_idx+1}.json", 'w') as f:
                json_lib.dump(fold_save_clean, f, indent=2, default=str)
            
            np.save(fold_dir / f"fold_{fold_idx+1}_mask.npy", feature_mask)
            
            print(f"    Test Acc: {eval_metrics['test_acc']*100:.2f}%, "
                  f"F1: {eval_metrics['test_f1_macro']*100:.2f}%, "
                  f"Features: {np.sum(feature_mask)}/{fold_total_features} "
                  f"({np.sum(feature_mask)/fold_total_features*100:.1f}%), "
                  f"Model: {eval_metrics['model_size_mb']:.3f}MB")
            
            # Cleanup
            del model, train_dataset, val_dataset, test_dataset
            del train_loader, val_loader, test_loader
            torch.cuda.empty_cache()
            gc.collect()
    
    print(f"\n{'='*80}")
    print("ALL PER-FOLD EXPERIMENTS COMPLETED")
    print(f"{'='*80}")
    
    return master_results

print("Per-fold experiment runner defined")
print("  -> Per-modality normalization: ENABLED")


Per-fold experiment runner defined
  -> Per-modality normalization: ENABLED


## Run All Experiments

In [10]:
master_results = run_all_methods_per_fold()

STARTING COMPREHENSIVE PER-FOLD EXPERIMENTS
Methods: baseline + 3 standard + 14 metaheuristics = 18 total
Folds: 10
Per-modality normalization: ENABLED

################################################################################
# FOLD 1/10
################################################################################
  Train: 405, Val: 27, Test: 184
    Feature dims after processing: sensor=4480 | skeleton=1879 | TOTAL=6359

  --- BASELINE ---
    Test Acc: 98.91%, F1: 98.92%, Features: 6359/6359 (100.0%), Model: 12.944MB

  --- MUTUAL_INFO ---
    Running Mutual Information...
      Selected: 3179/6359 (50.0%), Time: 34.1s
    Test Acc: 99.46%, F1: 99.46%, Features: 3179/6359 (50.0%), Model: 6.733MB

  --- RFE ---
    Running RFE...
      Selected: 3179/6359 (50.0%), Time: 8.5s
    Test Acc: 99.46%, F1: 99.46%, Features: 3179/6359 (50.0%), Model: 6.733MB

  --- LASSO ---
    Running LASSO...
      Selected: 3179/6359 (50.0%), Time: 110.1s
    Test Acc: 100.00%, F1: 100.00%, Fe

## Build Comprehensive Per-Fold Results CSV

In [11]:
# Build a single massive DataFrame with every fold x method combination
# Also extract FEATURE_DIMS from fold results for downstream plots
FEATURE_DIMS = None
TOTAL_FEATURES = None

rows = []

for method_name in ALL_METHODS:
    for fold_idx in range(N_FOLDS):
        if fold_idx not in master_results[method_name]:
            continue
        r = master_results[method_name][fold_idx]
        
        # Extract feature dims from first available result
        if FEATURE_DIMS is None and 'feature_dims' in r:
            FEATURE_DIMS = r['feature_dims']
            TOTAL_FEATURES = sum(FEATURE_DIMS.values())
            print(f"Feature dims: {FEATURE_DIMS}, Total: {TOTAL_FEATURES}")
        
        row = {
            'Method': method_name,
            'Fold': fold_idx + 1,
            'Test Accuracy (%)': r['test_acc'] * 100,
            'Val Accuracy (%)': r['val_acc'] * 100,
            'F1 Macro (%)': r['test_f1_macro'] * 100,
            'F1 Weighted (%)': r['test_f1_weighted'] * 100,
            'Precision Macro (%)': r['test_precision_macro'] * 100,
            'Recall Macro (%)': r['test_recall_macro'] * 100,
            'Features Selected': r['num_features_selected'],
            'Features Total': r['num_features_total'],
            'Feature Retention (%)': r['feature_retention_pct'],
            'FS Time (s)': r['fs_execution_time'],
            'Train Time (s)': r['train_time_sec'],
            'Total Time (s)': r['total_time_sec'],
            'Model Params': r['model_params'],
            'Model Size (MB)': r['model_size_mb'],
            'Orig Dataset (MB)': r['original_dataset_size_mb'],
            'Opt Dataset (MB)': r['optimized_dataset_size_mb'],
            'Dataset Reduction (%)': r['dataset_reduction_pct'],
            'GPU Peak (MB)': r['gpu_mem_peak_mb'],
            'N Train': r['num_train'],
            'N Val': r['num_val'],
            'N Test': r['num_test'],
        }
        
        # Per-modality retention
        for mod in MODALITY_NAMES:
            if mod in r.get('modality_retention', {}):
                row[f'{mod}_retained'] = r['modality_retention'][mod]['selected']
                row[f'{mod}_total'] = r['modality_retention'][mod]['total']
                row[f'{mod}_retention_%'] = r['modality_retention'][mod]['percentage']
        
        rows.append(row)

df_all = pd.DataFrame(rows)
df_all.to_csv(RESULTS_ROOT / "all_folds_results.csv", index=False)
print(f"Results table: {df_all.shape}")
print(f"Methods: {df_all['Method'].nunique()}, Folds per method: {df_all.groupby('Method')['Fold'].count().to_dict()}")
df_all.head()


Feature dims: {'sensor': 4480, 'skeleton': 1879}, Total: 6359
Results table: (180, 29)
Methods: 18, Folds per method: {'baseline': 10, 'lasso': 10, 'meta_BAT': 10, 'meta_CS': 10, 'meta_DE': 10, 'meta_FFA': 10, 'meta_GA': 10, 'meta_GWO': 10, 'meta_HHO': 10, 'meta_JAYA': 10, 'meta_MFO': 10, 'meta_MVO': 10, 'meta_PSO': 10, 'meta_SCA': 10, 'meta_SSA': 10, 'meta_WOA': 10, 'mutual_info': 10, 'rfe': 10}


,Method,Fold,Test Accuracy (%),Val Accuracy (%),F1 Macro (%),F1 Weighted (%),Precision Macro (%),Recall Macro (%),Features Selected,Features Total,...,GPU Peak (MB),N Train,N Val,N Test,sensor_retained,sensor_total,sensor_retention_%,skeleton_retained,skeleton_total,skeleton_retention_%
0,baseline,1,98.913043,100.0,98.917335,98.912247,98.973684,98.918129,6359,6359,...,94.676758,405,27,184,4480,4480,100.0,1879,1879,100.0
1,baseline,2,98.823529,100.0,98.776607,98.822421,98.848684,98.777778,6359,6359,...,94.676758,419,27,170,4480,4480,100.0,1879,1879,100.0
2,baseline,3,98.333333,100.0,98.321678,98.321678,98.571429,98.333333,6359,6359,...,94.676758,529,27,60,4480,4480,100.0,1879,1879,100.0
3,baseline,4,97.058824,100.0,97.460317,97.012138,98.000000,97.500000,6359,6359,...,94.676758,555,27,34,4480,4480,100.0,1879,1879,100.0
4,baseline,5,96.969697,100.0,96.571429,96.883117,97.500000,96.666667,6359,6359,...,94.676758,556,27,33,4480,4480,100.0,1879,1879,100.0


## Per-Method Summary Table (with per-fold detail)

In [12]:
# Also create a summary table (mean/std across folds)
summary_rows = []
for method_name in ALL_METHODS:
    method_df = df_all[df_all['Method'] == method_name]
    if len(method_df) == 0:
        continue
    
    row = {
        'Method': method_name,
        'Mean Test Acc (%)': method_df['Test Accuracy (%)'].mean(),
        'Std Test Acc (%)': method_df['Test Accuracy (%)'].std(),
        'Mean F1 Macro (%)': method_df['F1 Macro (%)'].mean(),
        'Std F1 Macro (%)': method_df['F1 Macro (%)'].std(),
        'Mean Features Selected': method_df['Features Selected'].mean(),
        'Mean Feature Retention (%)': method_df['Feature Retention (%)'].mean(),
        'Mean FS Time (s)': method_df['FS Time (s)'].mean(),
        'Mean Train Time (s)': method_df['Train Time (s)'].mean(),
        'Mean Total Time (s)': method_df['Total Time (s)'].mean(),
        'Mean Model Params': method_df['Model Params'].mean(),
        'Mean Model Size (MB)': method_df['Model Size (MB)'].mean(),
        'Mean Dataset Reduction (%)': method_df['Dataset Reduction (%)'].mean(),
        'Mean GPU Peak (MB)': method_df['GPU Peak (MB)'].mean(),
    }
    summary_rows.append(row)

df_summary = pd.DataFrame(summary_rows).round(2)
df_summary.to_csv(RESULTS_ROOT / "summary_all_methods.csv", index=False)
print("\nSUMMARY TABLE (mean across folds):")
print(df_summary.to_string(index=False))



SUMMARY TABLE (mean across folds):
     Method  Mean Test Acc (%)  Std Test Acc (%)  Mean F1 Macro (%)  Std F1 Macro (%)  Mean Features Selected  Mean Feature Retention (%)  Mean FS Time (s)  Mean Train Time (s)  Mean Total Time (s)  Mean Model Params  Mean Model Size (MB)  Mean Dataset Reduction (%)  Mean GPU Peak (MB)
   baseline              98.01              3.05              97.67              4.04                  6359.0                      100.00              0.00                21.48                21.48          3391754.0                 12.94                        0.00               94.68
mutual_info              98.11              3.16              97.81              4.13                  3179.0                       49.99             34.45                15.25                49.70          1763594.0                  6.73                       50.01               57.02
        rfe              97.53              3.22              96.95              4.19                  

## Comprehensive Visualizations

In [13]:
# ============================================================================
# PLOT 1: Test Accuracy per fold for all methods (heatmap)
# ============================================================================
pivot = df_all.pivot_table(values='Test Accuracy (%)', index='Method', columns='Fold')
pivot = pivot.reindex(ALL_METHODS)

fig, ax = plt.subplots(figsize=(14, 10))
sns.heatmap(pivot, annot=True, fmt='.1f', cmap='YlGnBu', ax=ax, linewidths=0.5, cbar_kws={'label': 'Test Accuracy (%)'})
ax.set_title('Test Accuracy (%) per Method per Fold', fontsize=16)
ax.set_ylabel('Method')
ax.set_xlabel('Fold')
plt.tight_layout()
plt.savefig(PLOTS_DIR / '01_accuracy_heatmap.png', dpi=300, bbox_inches='tight')
plt.close()
print("Saved: 01_accuracy_heatmap.png")


Saved: 01_accuracy_heatmap.png


In [14]:
# ============================================================================
# PLOT 2: Box plot of test accuracy across folds for each method
# ============================================================================
fig, ax = plt.subplots(figsize=(16, 7))
methods_sorted = df_summary.sort_values('Mean Test Acc (%)', ascending=False)['Method'].tolist()
order = methods_sorted

sns.boxplot(data=df_all, x='Method', y='Test Accuracy (%)', order=order, ax=ax, palette='Set2')
sns.stripplot(data=df_all, x='Method', y='Test Accuracy (%)', order=order, ax=ax, 
              color='black', alpha=0.5, size=4, jitter=True)
ax.set_title('Test Accuracy Distribution Across Folds', fontsize=16)
ax.set_xlabel('')
plt.xticks(rotation=60, ha='right')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(PLOTS_DIR / '02_accuracy_boxplot.png', dpi=300, bbox_inches='tight')
plt.close()
print("Saved: 02_accuracy_boxplot.png")


Saved: 02_accuracy_boxplot.png


In [15]:
# ============================================================================
# PLOT 3: Feature Retention per method (bar chart with error bars)
# ============================================================================
fig, ax = plt.subplots(figsize=(16, 7))
feat_summary = df_all.groupby('Method')['Feature Retention (%)'].agg(['mean', 'std']).reindex(ALL_METHODS)
bars = ax.bar(feat_summary.index, feat_summary['mean'], yerr=feat_summary['std'], capsize=3, alpha=0.7, color='coral')
ax.axhline(y=100, color='r', linestyle='--', label='All Features (100%)')
ax.set_ylabel('Feature Retention (%)', fontsize=12)
ax.set_title('Feature Retention by Method', fontsize=16)
ax.legend()
plt.xticks(rotation=60, ha='right')
ax.grid(axis='y', alpha=0.3)

# Value labels
for bar, mean_val in zip(bars, feat_summary['mean']):
    if not np.isnan(mean_val):
        ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 2,
                f'{mean_val:.1f}%', ha='center', va='bottom', fontsize=8, fontweight='bold')

plt.tight_layout()
plt.savefig(PLOTS_DIR / '03_feature_retention.png', dpi=300, bbox_inches='tight')
plt.close()
print("Saved: 03_feature_retention.png")


Saved: 03_feature_retention.png


In [16]:
# ============================================================================
# PLOT 4: Accuracy vs Feature Retention trade-off (scatter)
# ============================================================================
fig, ax = plt.subplots(figsize=(12, 8))

for method in ALL_METHODS:
    mdf = df_all[df_all['Method'] == method]
    if len(mdf) == 0:
        continue
    ax.scatter(mdf['Feature Retention (%)'], mdf['Test Accuracy (%)'], 
               label=method, s=60, alpha=0.7)
    # Draw mean point larger
    ax.scatter(mdf['Feature Retention (%)'].mean(), mdf['Test Accuracy (%)'].mean(),
               s=150, marker='X', edgecolors='black', linewidths=1.5)

ax.set_xlabel('Feature Retention (%)', fontsize=12)
ax.set_ylabel('Test Accuracy (%)', fontsize=12)
ax.set_title('Accuracy vs Feature Retention Trade-off', fontsize=16)
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(PLOTS_DIR / '04_accuracy_vs_retention.png', dpi=300, bbox_inches='tight')
plt.close()
print("Saved: 04_accuracy_vs_retention.png")


Saved: 04_accuracy_vs_retention.png


In [17]:
# ============================================================================
# PLOT 5: Per-modality retention heatmap (average across folds)
# ============================================================================
modality_data = {}
for method in ALL_METHODS:
    modality_data[method] = {}
    for mod in FEATURE_DIMS.keys():
        col = f'{mod}_retention (%)'
        if col in df_all.columns:
            mdf = df_all[df_all['Method'] == method]
            modality_data[method][mod] = mdf[col].mean() if len(mdf) > 0 else 0

mod_df = pd.DataFrame(modality_data).T
mod_df = mod_df.reindex(ALL_METHODS)

fig, ax = plt.subplots(figsize=(10, 10))
sns.heatmap(mod_df, annot=True, fmt='.1f', cmap='RdYlGn', ax=ax, linewidths=0.5,
            vmin=0, vmax=100, cbar_kws={'label': 'Retention (%)'})
ax.set_title('Average Modality-wise Feature Retention (%)', fontsize=14)
ax.set_ylabel('Method')
ax.set_xlabel('Modality')
plt.tight_layout()
plt.savefig(PLOTS_DIR / '05_modality_retention_heatmap.png', dpi=300, bbox_inches='tight')
plt.close()
print("Saved: 05_modality_retention_heatmap.png")


Saved: 05_modality_retention_heatmap.png


In [18]:
# ============================================================================
# PLOT 6: Execution time comparison
# ============================================================================
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# FS time
time_summary = df_all.groupby('Method')['FS Time (s)'].agg(['mean', 'std']).reindex(ALL_METHODS)
bars = axes[0].bar(time_summary.index, time_summary['mean'], yerr=time_summary['std'], capsize=3, alpha=0.7, color='steelblue')
axes[0].set_ylabel('Feature Selection Time (s)', fontsize=12)
axes[0].set_title('Feature Selection Time', fontsize=14)
plt.setp(axes[0].xaxis.get_majorticklabels(), rotation=60, ha='right')
axes[0].grid(axis='y', alpha=0.3)

# Total time
total_summary = df_all.groupby('Method')['Total Time (s)'].agg(['mean', 'std']).reindex(ALL_METHODS)
bars2 = axes[1].bar(total_summary.index, total_summary['mean'], yerr=total_summary['std'], capsize=3, alpha=0.7, color='darkorange')
axes[1].set_ylabel('Total Time (s)', fontsize=12)
axes[1].set_title('Total Time (FS + Training)', fontsize=14)
plt.setp(axes[1].xaxis.get_majorticklabels(), rotation=60, ha='right')
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(PLOTS_DIR / '06_execution_time.png', dpi=300, bbox_inches='tight')
plt.close()
print("Saved: 06_execution_time.png")


Saved: 06_execution_time.png


In [19]:
# ============================================================================
# PLOT 7: Model size & dataset size comparison
# ============================================================================
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# Model params
param_summary = df_all.groupby('Method')['Model Params'].mean().reindex(ALL_METHODS)
axes[0].bar(param_summary.index, param_summary.values, alpha=0.7, color='mediumpurple')
axes[0].set_ylabel('Model Parameters', fontsize=12)
axes[0].set_title('Average Model Parameters', fontsize=14)
plt.setp(axes[0].xaxis.get_majorticklabels(), rotation=60, ha='right')
axes[0].grid(axis='y', alpha=0.3)

# Dataset reduction
ds_summary = df_all.groupby('Method')['Dataset Reduction (%)'].mean().reindex(ALL_METHODS)
axes[1].bar(ds_summary.index, ds_summary.values, alpha=0.7, color='seagreen')
axes[1].set_ylabel('Dataset Size Reduction (%)', fontsize=12)
axes[1].set_title('Average Dataset Size Reduction', fontsize=14)
plt.setp(axes[1].xaxis.get_majorticklabels(), rotation=60, ha='right')
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(PLOTS_DIR / '07_model_dataset_size.png', dpi=300, bbox_inches='tight')
plt.close()
print("Saved: 07_model_dataset_size.png")


Saved: 07_model_dataset_size.png


In [20]:
# ============================================================================
# PLOT 8: Metaheuristics-only comparison (grouped bar: accuracy & retention)
# ============================================================================
meta_methods = [m for m in ALL_METHODS if m.startswith('meta_')]

if len(meta_methods) > 0:
    meta_df = df_all[df_all['Method'].isin(meta_methods)]
    
    fig, axes = plt.subplots(2, 1, figsize=(16, 12))
    
    # Accuracy
    meta_acc = meta_df.groupby('Method')['Test Accuracy (%)'].agg(['mean', 'std']).reindex(meta_methods)
    bars = axes[0].bar(range(len(meta_methods)), meta_acc['mean'], yerr=meta_acc['std'], 
                        capsize=4, alpha=0.7, color=plt.cm.tab20(np.linspace(0, 1, len(meta_methods))))
    axes[0].set_xticks(range(len(meta_methods)))
    axes[0].set_xticklabels([m.replace('meta_', '') for m in meta_methods], rotation=45, ha='right')
    axes[0].set_ylabel('Test Accuracy (%)')
    axes[0].set_title('Metaheuristics: Test Accuracy Comparison', fontsize=14)
    axes[0].grid(axis='y', alpha=0.3)
    
    for i, (bar, val) in enumerate(zip(bars, meta_acc['mean'])):
        if not np.isnan(val):
            axes[0].text(bar.get_x() + bar.get_width()/2., bar.get_height() + meta_acc['std'].iloc[i] + 0.5,
                        f'{val:.1f}', ha='center', va='bottom', fontsize=8)
    
    # Feature retention
    meta_feat = meta_df.groupby('Method')['Feature Retention (%)'].agg(['mean', 'std']).reindex(meta_methods)
    bars2 = axes[1].bar(range(len(meta_methods)), meta_feat['mean'], yerr=meta_feat['std'],
                         capsize=4, alpha=0.7, color=plt.cm.tab20(np.linspace(0, 1, len(meta_methods))))
    axes[1].set_xticks(range(len(meta_methods)))
    axes[1].set_xticklabels([m.replace('meta_', '') for m in meta_methods], rotation=45, ha='right')
    axes[1].set_ylabel('Feature Retention (%)')
    axes[1].set_title('Metaheuristics: Feature Retention Comparison', fontsize=14)
    axes[1].grid(axis='y', alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(PLOTS_DIR / '08_metaheuristics_comparison.png', dpi=300, bbox_inches='tight')
    plt.close()
    print("Saved: 08_metaheuristics_comparison.png")


Saved: 08_metaheuristics_comparison.png


In [21]:
# ============================================================================
# PLOT 9: Convergence curves for metaheuristics (average across folds)
# ============================================================================
meta_methods = [m for m in ALL_METHODS if m.startswith('meta_')]

if len(meta_methods) > 0:
    fig, ax = plt.subplots(figsize=(14, 8))
    
    for method in meta_methods:
        all_conv = []
        for fold_idx in range(N_FOLDS):
            if fold_idx in master_results[method]:
                conv = master_results[method][fold_idx].get('convergence', None)
                if conv is not None:
                    all_conv.append(conv)
        
        if all_conv:
            # Pad to same length
            max_len = max(len(c) for c in all_conv)
            padded = []
            for c in all_conv:
                if len(c) < max_len:
                    c = list(c) + [c[-1]] * (max_len - len(c))
                padded.append(c)
            
            avg_conv = np.mean(padded, axis=0)
            ax.plot(avg_conv, label=method.replace('meta_', ''), linewidth=1.5)
    
    ax.set_xlabel('Iteration', fontsize=12)
    ax.set_ylabel('Fitness (1 - Accuracy)', fontsize=12)
    ax.set_title('Metaheuristic Convergence Curves (averaged across folds)', fontsize=14)
    ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(PLOTS_DIR / '09_convergence_curves.png', dpi=300, bbox_inches='tight')
    plt.close()
    print("Saved: 09_convergence_curves.png")


Saved: 09_convergence_curves.png


In [22]:
# ============================================================================
# PLOT 10: Feature importance - which features most commonly retained/removed
# ============================================================================
# Count how many times each feature index was selected across all methods and folds
feature_freq = np.zeros(TOTAL_FEATURES)
feature_freq_per_method = {}

for method in ALL_METHODS:
    if method == 'baseline':
        continue
    method_freq = np.zeros(TOTAL_FEATURES)
    count = 0
    for fold_idx in range(N_FOLDS):
        if fold_idx in master_results[method]:
            mask = master_results[method][fold_idx]['feature_mask']
            feature_freq += mask.astype(float)
            method_freq += mask.astype(float)
            count += 1
    if count > 0:
        feature_freq_per_method[method] = method_freq / count

# Normalize overall
total_runs = (len(ALL_METHODS) - 1) * N_FOLDS  # exclude baseline
feature_freq_normalized = feature_freq / max(total_runs, 1)

# Plot feature selection frequency
fig, axes = plt.subplots(2, 1, figsize=(18, 10))

# Overall
axes[0].bar(range(TOTAL_FEATURES), feature_freq_normalized, width=1.0, alpha=0.7, color='steelblue')
axes[0].set_xlabel('Feature Index')
axes[0].set_ylabel('Selection Frequency')
axes[0].set_title('Feature Selection Frequency Across All Methods and Folds', fontsize=14)

# Add modality boundaries
start = 0
colors = ['red', 'green', 'blue', 'purple']
for i, (mod, dim) in enumerate(FEATURE_DIMS.items()):
    axes[0].axvline(x=start, color=colors[i], linestyle='--', alpha=0.5, label=f'{mod} start')
    start += dim
axes[0].legend(fontsize=8)
axes[0].grid(axis='y', alpha=0.3)

# Top 20 most and least selected features
top_20_selected = np.argsort(feature_freq_normalized)[::-1][:20]
top_20_removed = np.argsort(feature_freq_normalized)[:20]

x_pos = np.arange(20)
width = 0.35
axes[1].bar(x_pos - width/2, feature_freq_normalized[top_20_selected], width, label='Top 20 Most Selected', color='green', alpha=0.7)
axes[1].bar(x_pos + width/2, feature_freq_normalized[top_20_removed], width, label='Top 20 Least Selected', color='red', alpha=0.7)
axes[1].set_xlabel('Rank')
axes[1].set_ylabel('Selection Frequency')
axes[1].set_title('Most vs Least Frequently Selected Features', fontsize=14)
axes[1].legend()
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(PLOTS_DIR / '10_feature_frequency.png', dpi=300, bbox_inches='tight')
plt.close()
print("Saved: 10_feature_frequency.png")

# Save feature frequency data
freq_df = pd.DataFrame({
    'feature_index': range(TOTAL_FEATURES),
    'overall_selection_freq': feature_freq_normalized,
})
# Add modality labels
start = 0
modality_labels = []
for mod, dim in FEATURE_DIMS.items():
    modality_labels.extend([mod] * dim)
freq_df['modality'] = modality_labels
freq_df.to_csv(PLOTS_DIR / 'feature_selection_frequency.csv', index=False)
print("Saved: feature_selection_frequency.csv")


Saved: 10_feature_frequency.png
Saved: feature_selection_frequency.csv


In [23]:
# ============================================================================
# PLOT 11: Radar/spider chart comparing methods on multiple metrics
# ============================================================================
from matplotlib.patches import FancyBboxPatch
import matplotlib.patches as mpatches

# Select key metrics for radar
radar_metrics = ['Mean Test Acc (%)', 'Mean F1 Macro (%)', 'Mean Feature Retention (%)', 'Mean Total Time (s)']

# Normalize to [0, 1] for radar
radar_data = df_summary[['Method'] + radar_metrics].copy()
for col in radar_metrics:
    min_val = radar_data[col].min()
    max_val = radar_data[col].max()
    if max_val > min_val:
        radar_data[col] = (radar_data[col] - min_val) / (max_val - min_val)
    else:
        radar_data[col] = 0.5

# For time, invert (lower is better)
radar_data['Mean Total Time (s)'] = 1 - radar_data['Mean Total Time (s)']
radar_data = radar_data.rename(columns={'Mean Total Time (s)': 'Speed (inv. time)'})
radar_cols = ['Mean Test Acc (%)', 'Mean F1 Macro (%)', 'Mean Feature Retention (%)', 'Speed (inv. time)']

# Select subset for readability (standard + top 5 metaheuristics)
top_meta = df_summary[df_summary['Method'].str.startswith('meta_')].nlargest(5, 'Mean Test Acc (%)')['Method'].tolist()
methods_to_plot = ['baseline', 'mutual_info', 'rfe', 'lasso'] + top_meta

fig, ax = plt.subplots(figsize=(10, 10), subplot_kw=dict(polar=True))
angles = np.linspace(0, 2 * np.pi, len(radar_cols), endpoint=False).tolist()
angles += angles[:1]

for method in methods_to_plot:
    row = radar_data[radar_data['Method'] == method]
    if len(row) == 0:
        continue
    values = row[radar_cols].values.flatten().tolist()
    values += values[:1]
    ax.plot(angles, values, linewidth=1.5, label=method)
    ax.fill(angles, values, alpha=0.05)

ax.set_xticks(angles[:-1])
ax.set_xticklabels(radar_cols, fontsize=9)
ax.set_title('Multi-Metric Comparison (Normalized)', fontsize=14, pad=20)
ax.legend(bbox_to_anchor=(1.3, 1.1), fontsize=8)
plt.tight_layout()
plt.savefig(PLOTS_DIR / '11_radar_comparison.png', dpi=300, bbox_inches='tight')
plt.close()
print("Saved: 11_radar_comparison.png")


Saved: 11_radar_comparison.png


In [24]:
# ============================================================================
# PLOT 12: Standard vs Metaheuristics grouped comparison
# ============================================================================
standard_methods = ['baseline', 'mutual_info', 'rfe', 'lasso']
meta_methods = [m for m in ALL_METHODS if m.startswith('meta_')]

fig, axes = plt.subplots(1, 3, figsize=(20, 7))

# Group averages
std_accs = df_all[df_all['Method'].isin(standard_methods)].groupby('Method')['Test Accuracy (%)'].mean()
meta_accs = df_all[df_all['Method'].isin(meta_methods)].groupby('Method')['Test Accuracy (%)'].mean()

# Accuracy: standard vs meta
all_std = df_all[df_all['Method'].isin(standard_methods)]['Test Accuracy (%)']
all_meta = df_all[df_all['Method'].isin(meta_methods)]['Test Accuracy (%)']
bp = axes[0].boxplot([all_std, all_meta], labels=['Standard\n(MI, RFE, LASSO)', 'Metaheuristics\n(14 optimizers)'],
                      patch_artist=True)
bp['boxes'][0].set_facecolor('lightblue')
bp['boxes'][1].set_facecolor('lightsalmon')
axes[0].set_ylabel('Test Accuracy (%)')
axes[0].set_title('Standard vs Metaheuristic Accuracy', fontsize=13)
axes[0].grid(axis='y', alpha=0.3)

# Feature retention
all_std_f = df_all[(df_all['Method'].isin(standard_methods)) & (df_all['Method'] != 'baseline')]['Feature Retention (%)']
all_meta_f = df_all[df_all['Method'].isin(meta_methods)]['Feature Retention (%)']
bp2 = axes[1].boxplot([all_std_f, all_meta_f], labels=['Standard', 'Metaheuristics'],
                       patch_artist=True)
bp2['boxes'][0].set_facecolor('lightblue')
bp2['boxes'][1].set_facecolor('lightsalmon')
axes[1].set_ylabel('Feature Retention (%)')
axes[1].set_title('Standard vs Metaheuristic Feature Retention', fontsize=13)
axes[1].grid(axis='y', alpha=0.3)

# Time
all_std_t = df_all[(df_all['Method'].isin(standard_methods)) & (df_all['Method'] != 'baseline')]['Total Time (s)']
all_meta_t = df_all[df_all['Method'].isin(meta_methods)]['Total Time (s)']
bp3 = axes[2].boxplot([all_std_t, all_meta_t], labels=['Standard', 'Metaheuristics'],
                       patch_artist=True)
bp3['boxes'][0].set_facecolor('lightblue')
bp3['boxes'][1].set_facecolor('lightsalmon')
axes[2].set_ylabel('Total Time (s)')
axes[2].set_title('Standard vs Metaheuristic Time', fontsize=13)
axes[2].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(PLOTS_DIR / '12_standard_vs_metaheuristics.png', dpi=300, bbox_inches='tight')
plt.close()
print("Saved: 12_standard_vs_metaheuristics.png")


Saved: 12_standard_vs_metaheuristics.png


In [25]:
# ============================================================================
# PLOT 13: GPU memory usage comparison
# ============================================================================
fig, ax = plt.subplots(figsize=(16, 7))
gpu_summary = df_all.groupby('Method')['GPU Peak (MB)'].agg(['mean', 'std']).reindex(ALL_METHODS)
bars = ax.bar(gpu_summary.index, gpu_summary['mean'], yerr=gpu_summary['std'], capsize=3, alpha=0.7, color='gold')
ax.set_ylabel('GPU Peak Memory (MB)', fontsize=12)
ax.set_title('GPU Peak Memory Usage by Method', fontsize=16)
plt.xticks(rotation=60, ha='right')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(PLOTS_DIR / '13_gpu_memory.png', dpi=300, bbox_inches='tight')
plt.close()
print("Saved: 13_gpu_memory.png")


Saved: 13_gpu_memory.png


In [26]:
# ============================================================================
# PLOT 14: Per-fold accuracy line plot (all methods)
# ============================================================================
fig, ax = plt.subplots(figsize=(16, 8))

for method in ALL_METHODS:
    mdf = df_all[df_all['Method'] == method].sort_values('Fold')
    if len(mdf) > 0:
        linewidth = 2.5 if method in ['baseline', 'mutual_info', 'rfe', 'lasso'] else 1.0
        alpha = 1.0 if method in ['baseline', 'mutual_info', 'rfe', 'lasso'] else 0.6
        ax.plot(mdf['Fold'], mdf['Test Accuracy (%)'], marker='o', markersize=4,
                label=method, linewidth=linewidth, alpha=alpha)

ax.set_xlabel('Fold', fontsize=12)
ax.set_ylabel('Test Accuracy (%)', fontsize=12)
ax.set_title('Test Accuracy Across Folds (All Methods)', fontsize=16)
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8)
ax.grid(alpha=0.3)
ax.set_xticks(range(1, N_FOLDS + 1))
plt.tight_layout()
plt.savefig(PLOTS_DIR / '14_per_fold_accuracy_lines.png', dpi=300, bbox_inches='tight')
plt.close()
print("Saved: 14_per_fold_accuracy_lines.png")


Saved: 14_per_fold_accuracy_lines.png


In [27]:
# ============================================================================
# BEST METAHEURISTIC PER FOLD: F1 Macro
# ============================================================================

df_all = pd.read_csv(RESULTS_ROOT / "all_folds_results.csv")

meta_methods   = sorted([m for m in df_all['Method'].unique() if m.startswith('meta_')])
fold_ids       = sorted(df_all['Fold'].unique())

print("=" * 90)
print("BEST METAHEURISTIC PER FOLD (selected by highest F1 Macro)")
print("=" * 90)

best_per_fold = []

for fold in fold_ids:
    fold_df = df_all[(df_all['Fold'] == fold) & (df_all['Method'].isin(meta_methods))]
    if fold_df.empty:
        print(f"\n  Fold {fold}: No metaheuristic results found!")
        continue

    best_row = fold_df.loc[fold_df['F1 Macro (%)'].idxmax()]

    entry = {
        'Fold':                   fold,
        'Best Metaheuristic':     best_row['Method'],
        'F1 Macro (%)':           best_row['F1 Macro (%)'],
        'F1 Weighted (%)':        best_row['F1 Weighted (%)'],
        'Test Accuracy (%)':      best_row['Test Accuracy (%)'],
        'Feature Retention (%)':  best_row['Feature Retention (%)'],
        'Features Selected':      best_row['Features Selected'],
        'Features Total':         best_row['Features Total'],
    }

    # Per-modality retention — grab whatever *_retention_% columns exist
    mod_cols = [c for c in df_all.columns if c.endswith('_retention_%')]
    for col in mod_cols:
        entry[col] = best_row[col]

    best_per_fold.append(entry)

    mod_str = ', '.join([f"{c.replace('_retention_%','')}={best_row[c]:.1f}%"
                         for c in mod_cols])
    print(f"\n  Fold {fold}: Best = {best_row['Method']}")
    print(f"    F1 Macro:           {entry['F1 Macro (%)']:.2f}%")
    print(f"    F1 Weighted:        {entry['F1 Weighted (%)']:.2f}%")
    print(f"    Test Accuracy:      {entry['Test Accuracy (%)']:.2f}%")
    print(f"    Feature Retention:  {entry['Feature Retention (%)']:.2f}%"
          f" ({int(entry['Features Selected'])}/{int(entry['Features Total'])})")
    if mod_str:
        print(f"    Modality Retention: {mod_str}")

# ============================================================================
# AVERAGE ACROSS FOLDS
# ============================================================================
df_best = pd.DataFrame(best_per_fold)

print("\n" + "=" * 90)
print("AVERAGE ACROSS ALL FOLDS (Best Metaheuristic per Fold)")
print("=" * 90)

for col, label in [
    ('F1 Macro (%)',          'Average F1 Macro'),
    ('F1 Weighted (%)',       'Average F1 Weighted'),
    ('Test Accuracy (%)',     'Average Test Accuracy'),
    ('Feature Retention (%)', 'Average Feature Retention'),
]:
    print(f"\n  {label:28s}: {df_best[col].mean():.2f}% ± {df_best[col].std():.2f}%")

mod_cols = [c for c in df_best.columns if c.endswith('_retention_%')]
if mod_cols:
    print(f"\n  Average Modality-wise Feature Retention:")
    for col in mod_cols:
        mod = col.replace('_retention_%', '')
        print(f"    {mod:>12s}: {df_best[col].mean():.2f}% ± {df_best[col].std():.2f}%")

# ============================================================================
# SUMMARY TABLE
# ============================================================================
print("\n" + "=" * 90)
print("PER-FOLD SUMMARY TABLE")
print("=" * 90)

display_cols = ['Fold', 'Best Metaheuristic', 'F1 Macro (%)', 'F1 Weighted (%)',
                'Test Accuracy (%)', 'Feature Retention (%)'] + mod_cols
print(df_best[display_cols].to_string(index=False, float_format='%.2f'))

csv_path = RESULTS_ROOT / "best_metaheuristic_per_fold.csv"
df_best.to_csv(csv_path, index=False)
print(f"\nSaved to: {csv_path}")

BEST METAHEURISTIC PER FOLD (selected by highest F1 Macro)

  Fold 1: Best = meta_GA
    F1 Macro:           100.00%
    F1 Weighted:        100.00%
    Test Accuracy:      100.00%
    Feature Retention:  47.11% (2996/6359)
    Modality Retention: sensor=47.8%, skeleton=45.5%

  Fold 2: Best = meta_BAT
    F1 Macro:           100.00%
    F1 Weighted:        100.00%
    Test Accuracy:      100.00%
    Feature Retention:  48.44% (3080/6359)
    Modality Retention: sensor=48.1%, skeleton=49.2%

  Fold 3: Best = meta_MFO
    F1 Macro:           100.00%
    F1 Weighted:        100.00%
    Test Accuracy:      100.00%
    Feature Retention:  48.18% (3064/6359)
    Modality Retention: sensor=47.8%, skeleton=49.0%

  Fold 4: Best = meta_BAT
    F1 Macro:           97.46%
    F1 Weighted:        97.01%
    Test Accuracy:      97.06%
    Feature Retention:  47.99% (3052/6359)
    Modality Retention: sensor=48.5%, skeleton=46.8%

  Fold 5: Best = meta_DE
    F1 Macro:           100.00%
    F1 Weig